In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from datetime import date
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from torchvision import datasets, transforms

import copy
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import tqdm




def save_my_work(model, logfile, xdata_name, opt, Bsize, learningRate, epoch, second_time, best_mse, mse, loss, init_time):
    #model_save_time = str(int(time.time()))
    model_save_time = str(int(second_time))
    # model_file_name = '/content/drive/MyDrive/0_318lab/ML/NewJChem2021_45_9427/NewJChem2021_45_9427_git/results/'+'model' + model_save_time + '.pt'
    model_file_name = '/content/drive/MyDrive/0_318lab/ML/NewJChem2021_45_9427/NewJChem2021_45_9427_git/results/'+'model' + str(int(second_time)) + '_RMSE_' + str( ((mse**(1/2))//0.0001)*0.0001 ) + '.pt'
    elapsed_time = (time.time()-second_time)/60
    print('start to save model')
    torch.save(model, model_file_name )
    print('start to save log')
    print(
                    model_file_name,
                    'data:', xdata_name,
                    file=logfile
    )
    print(
                    ', layer_dim_list:', dim_list,
                    'activateion:', act,
                    ', optimizer:', opt,
                    ', Bsize:', Bsize ,
                    ', learningRate:', learningRate ,
                    file=logfile
                    )
    print(
                ', epoch: %5d' % epoch,
                ', passed_time: %.3f' % ( ( ( time.time()-second_time ) )  / 60  ),  'm',

                ", minimum_RMSE: %.5f" % ( best_mse**(1/2) ),
                ', RMSE_Test: %5f' % (mse**(1/2)),
                ', RMSE_Train: %5f' % (loss**(1/2)),
                ', passed_time_accum: %.3f' % ( ( time.time() -init_time )  / 60  ),  'm',
                'model_save_time' , model_save_time,
                file=logfile
                )


    print(
                    model_file_name,
                    'data:', xdata_name,
    )
    print(
                    ', layer_dim_list:', dim_list,
                    'activateion:', act,
                    ', optimizer:', opt,
                    ', Bsize:', Bsize ,
                    ', learningRate:', learningRate ,
                    )
    print(
                ', epoch: %5d' % epoch,
                ', passed_time: %.3f' % ( ( ( time.time()-second_time ) )  / 60  ),  'm',

                ", minimum_RMSE: %.5f" % (best_mse**(1/2)),
                ', test_loss: %5f' % (mse**(1/2)),
                ', train_loss: %5f' % (loss**(1/2)),
                ', passed_time_accum: %.3f' % ( ( time.time() -init_time )  / 60  ),  'm',
                'model_save_time' , model_save_time,
                )
    print('start to save figures')

    fig, axs = plt.subplots(2, 1, figsize=(5, 5*2))#, sharey=True)
    axs[0].set_xlabel('Epoch', fontsize=10)
    axs[0].set_ylabel(r'Test Loss', fontsize=10)
    axs[0].legend()
    axs[0].plot(history, )
    axs[0].set_yscale('log')
    axs[0].set_title('Test_Loss')
    axs[0].legend("RMSE:"+str( mse**(1/2) )+ "_"+str(act)+ "_"+str(opt)+"_"+str(Bsize)+"_"+ str(learningRate) +'_'+str((elapsed_time//0.001) * 0.001  ))


    axs[1].set_xlabel('Epoch', fontsize=10)
    axs[1].set_ylabel(r'Train Loss', fontsize=10)
    axs[1].legend()
    axs[1].plot(history_train)
    axs[1].set_yscale('log')
    axs[1].set_title('Train_Loss')

    # fig.legend(l1, ( str(act)+ "_"+str(opt)+"_"+str(Bsize)+"_"+ str(learningRate) +'_'+str(elapsed_time) ) )#, loc='upper left')
    # fig.legend(l2, ( str(act)+ "_"+str(opt)+"_"+str(Bsize)+"_"+ str(learningRate) +'_'+str(elapsed_time) ) )#, loc='upper right')

    # plt.legend([str(act)+ "_"+str(opt)+"_"+str(Bsize)+"_"+ str(learningRate) +'_'+str(elapsed_time)])#, fontsize="x-large")

    plt.savefig(
                model_file_name[0:-3] + 'Loss' + '.pdf',
                format="pdf",
                bbox_inches="tight"
                )
    plt.show()
    print('saving work is done')

def print_progress(act, opt, batch_size, learningRate, epoch, mse, loss, second_time):
    print(

        'act: ', act ,
        'opt: ', opt ,
        'batch_size: ', batch_size ,
        'learningRate: ', learningRate ,
        'epoch: %5d' % epoch,
        ', RMSE_Test: {:7.5f}'.format(mse**(1/2)),
        ', RMSE_Train: {:7.5f}'.format(loss**(1/2)),
        ', est_time: {:5.1f}'.format(( ( ( time.time()-second_time ) )  / 60  )),  'min,',
        'average_time: {:.2f}'.format( ( time.time()-second_time )/(epoch+1)), 's'
        )

In [13]:
def find_first_num(a):
    # how_many = 1
    len_a = len(a)
    # for n in range(100):
    #     try:
    #         how_many = int(a.split(pd_read_atoms["symboll"][n])[0])
    #         #print(i,j,k,n,how_many)
    #         break
    #     except:
    #         pass
    m = 0
    for n in range(0,len_a):
        try:
            how_many = int(a[0:n+1])
            #print(i,j,k,n,how_many)
            m=m+1
                
        except:
            break
    
    return a[0:m]

def count_element_num(a):
    ratio = {}

    for i in range(100):
        ratio[pd_read_atoms["symboll"][i]]=0
        k = a.split(pd_read_atoms["symboll"][i])
        # print(k)
        for j in range(len(k)):
            # print(j)
            if k[j]==a:
                pass
            elif j==0:
                pass
            else:
                print(pd_read_atoms["symboll"][i])
                print(k)
                print(k[j])
                print(find_first_num(k[j]))
                if find_first_num(k[j])=='':
                    ratio[pd_read_atoms["symboll"][i]]=ratio[pd_read_atoms["symboll"][i]]+1
                else:
                    ratio[pd_read_atoms["symboll"][i]]=ratio[pd_read_atoms["symboll"][i]]+int(find_first_num(k[j]))
                a = a.replace(pd_read_atoms["symboll"][i] + str(find_first_num(k[j])) ,'') 
                
                print(a)
                
                print("")
                #print('number:',pd_read_atoms["NUMBER"][i],'symboll:',pd_read_atoms["symboll"][i],',',k[j])
    return ratio

def where_as_list(a,b):
    # a= '[(CH3(CH2)5NH3)2(CH3NH3)2]Pb3I10'
    len_a=len(a)
    where_is_parenthesis = []
    # where_is_close = []
    for i in range(len_a):
        # if a[i]=='(' or a[i]=='[':
        #     where_is_open.append(i)
        # elif a[i]==')' or a[i]==']':
        #     where_is_close.append(i)
        # else:
        #     pass
        if a[i]=='(' or a[i]=='[':
            where_is_parenthesis.append(1)
        elif a[i]==')' or a[i]==']':
            where_is_parenthesis.append(2)
            if b==1:
                break
        else:
            where_is_parenthesis.append(0)
    return where_is_parenthesis

def where_as_tuple(where_is_parenthesis,b):
    whereAreThey = []
    len_a = len(where_is_parenthesis)
    for i in range(len_a):
        if where_is_parenthesis[i]==2:
            # where_is_parenthesis.pop(i)
            where_is_parenthesis[i]=0
            # print(i,end=' ')
            for j in range(i-1,-1,-1):
                if where_is_parenthesis[j]==1:
                    # where_is_parenthesis.pop(j)
                    where_is_parenthesis[j]=0
                    # print(j)
                    whereAreThey.append( (j,i) )
                    
                        
                    break
            if b==1:
                break
    return whereAreThey

def rewrite_formula(a):
    # e=a
    b = where_as_tuple( where_as_list(a,1),1 ) 
    # print(b)
    for i in b:
        c=a[i[0]+1:i[1]]
        d=a[i[1]+1:]
        # print(a)
        # print(c)
        count_element_num_a_sum = count_element_num(c)
        # for i in count_element_num_a_sum:
        #     if count_element_num_a_sum[i]!=0:
        #         print(i, count_element_num_a_sum[i])

        # print()
        # print(d)
        if find_first_num(d)=='':
            multiplier = 1
        else:
            multiplier = int(find_first_num(d))
        # print(multiplier)
        remove_this = c+str(multiplier)

        for j in count_element_num_a_sum:
            count_element_num_a_sum[j]=count_element_num_a_sum[j]*multiplier
        new = ''
        # len_remove_this = len(remove_this)
        for j in count_element_num_a_sum:
            if count_element_num_a_sum[j]!=0:
                # print(j, count_element_num_a_sum[j])
                new = new + j + str(count_element_num_a_sum[j])
        # print(a)
        if multiplier ==1:
            e = a[: i[0] ] + new + a[ i[0]+ len(remove_this)+1  : ]
        else:
            e = a[: i[0] ] + new + a[ i[0]+ len(remove_this)+2  : ]
        # print(e)
        # print(how_many)
        # print('')
    return e

In [3]:
# data_path = '/content/drive/MyDrive/0_318lab/ML/NewJChem2021_45_9427/inputData2.xlsx'
data_path = '../inputData2.xlsx'
data_path_atoms = '../atoms.xlsx'

pd_read = pd.read_excel(data_path)
pd_read_atoms = pd.read_excel(data_path_atoms)

x_np_array = np.array([])
y_np_array = np.array([])

pd_read_keys = pd_read.keys()
pd_read_keys_atoms = pd_read_atoms.keys()

len_pd_read_keys = len(pd_read.keys())
len_pd_read_keys_atoms = len(pd_read_atoms.keys())




In [8]:
pd_read

,Compound,"Optical Eg (exp.), eV"
0,[C5H11NCH2NH3][CH3NH3]6Pb7I22,1.53
1,[C(NH2)3][CH3NH3]3Pb3I10,1.56
2,[CH3(CH2)3NH3]2[CH3NH3]6Pb7I22,1.57
3,[CH3(CH2)2NH3]2[CH3NH3]2Pb3I10,1.58
4,[CH3(CH2)3NH3]2[CH3NH3]5Pb6I19,1.59
...,...,...
193,[(CH3)2S(CH2)4S(CH3)2]Pb3Br12,3.78
194,[NH3(CH2)2(CH3)(CH2)3NH3]PbCl4,3.87
195,[CH3NH3]2CdCl4,4.13
196,[NH3CH2CH(OH)CH2NH3]CdCl4,4.40


In [116]:
pd_read_atoms

,NUMBER,symboll,NAME,WEIGHT,1s,2s,2p,3s,3p,3d,...,5s,5p,5d,5f,6s,6p,6d,7s,Ionization Energy [eV],how_many
0,40,Zr,Zirconium,91.22000,2,2,6,2,6,10,...,2,0,0,0,0,0,0,0,6.634,0
1,30,Zn,Zinc,65.38000,2,2,6,2,6,10,...,0,0,0,0,0,0,0,0,9.394,0
2,70,Yb,Ytterbium,173.04000,2,2,6,2,6,10,...,2,6,0,0,2,0,0,0,6.254,0
3,39,Y,Yttrium,88.90590,2,2,6,2,6,10,...,2,0,0,0,0,0,0,0,6.217,0
4,54,Xe,Xenon,131.30000,2,2,6,2,6,10,...,2,6,0,0,0,0,0,0,12.130,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,18,Ar,Argon,39.94800,2,2,6,2,6,0,...,0,0,0,0,0,0,0,0,15.760,0
96,95,Am,Americium,243.00000,2,2,6,2,6,10,...,2,6,10,7,2,6,0,2,5.970,0
97,13,Al,Aluminum,26.98154,2,2,6,2,1,0,...,0,0,0,0,0,0,0,0,5.986,0
98,47,Ag,Silver,107.86800,2,2,6,2,6,10,...,1,0,0,0,0,0,0,0,7.576,0


In [9]:
 for k in range(len(pd_read)):
    
    k=2#-------------------------------delete
    print(k)
    
    a = pd_read['Compound'][k].replace("_x000D_\n",'')
    
    # compound_list = []

    while(1):
        if ('[' in a) or  ('(' in a):
            print(a)
            a = rewrite_formula(a)
        else:
            print(a)
            count_element_num_a_sum = count_element_num(a)
            for i in count_element_num_a_sum:
                if count_element_num_a_sum[i]!=0:
                    print(i, count_element_num_a_sum[i])

            break
    print('')
    break

2
[CH3(CH2)3NH3]2[CH3NH3]6Pb7I22
[CH3H6C3NH3]2[CH3NH3]6Pb7I22
N2H24C2[CH3NH3]6Pb7I22
N2H24C2N6H36C6Pb7I22
Pb 7
N 8
I 22
H 60
C 8



In [15]:
print(find_first_num('3'))

3


In [14]:
count_element_num('CH3H6C3NH3')

N
['CH3H6C3', 'H3']
H3

CH3H6C3H3

H
['C', '3', '6C3', '3']
3
3
CH6C3

H
['C', '3', '6C3', '3']
6C3
6
CC3

H
['C', '3', '6C3', '3']
3
3
CC3

C
['', '', '3']


3



{'Zr': 0,
 'Zn': 0,
 'Yb': 0,
 'Y': 0,
 'Xe': 0,
 'W': 0,
 'V': 0,
 'U': 0,
 'Tm': 0,
 'Tl': 0,
 'Ti': 0,
 'Th': 0,
 'Te': 0,
 'Tc': 0,
 'Tb': 0,
 'Ta': 0,
 'Sr': 0,
 'Sn': 0,
 'Sm': 0,
 'Si': 0,
 'Se': 0,
 'Sc': 0,
 'Sb': 0,
 'S': 0,
 'Ru': 0,
 'Rn': 0,
 'Rh': 0,
 'Re': 0,
 'Rb': 0,
 'Ra': 0,
 'Pu': 0,
 'Pt': 0,
 'Pr': 0,
 'Po': 0,
 'Pm': 0,
 'Pd': 0,
 'Pb': 0,
 'Pa': 0,
 'P': 0,
 'Os': 0,
 'O': 0,
 'Np': 0,
 'Ni': 0,
 'Ne': 0,
 'Nd': 0,
 'Nb': 0,
 'Na': 0,
 'N': 1,
 'Mo': 0,
 'Mn': 0,
 'Mg': 0,
 'Lu': 0,
 'Li': 0,
 'La': 0,
 'Kr': 0,
 'K': 0,
 'Ir': 0,
 'In': 0,
 'I': 0,
 'Ho': 0,
 'Hg': 0,
 'Hf': 0,
 'He': 0,
 'H': 12,
 'Ge': 0,
 'Gd': 0,
 'Ga': 0,
 'Fr': 0,
 'Fm': 0,
 'Fe': 0,
 'F': 0,
 'Eu': 0,
 'Es': 0,
 'Er': 0,
 'Dy': 0,
 'Cu': 0,
 'Cs': 0,
 'Cr': 0,
 'Co': 0,
 'Cm': 0,
 'Cl': 0,
 'Cf': 0,
 'Ce': 0,
 'Cd': 0,
 'Ca': 0,
 'C': 1,
 'Br': 0,
 'Bk': 0,
 'Bi': 0,
 'Be': 0,
 'Ba': 0,
 'B': 0,
 'Au': 0,
 'At': 0,
 'As': 0,
 'Ar': 0,
 'Am': 0,
 'Al': 0,
 'Ag': 0,
 'Ac': 0}

In [121]:
len_remove_this = len(remove_this)
for i in range(len_remove_this+2):
    a[b[0]]=''
a


TypeError: 'str' object does not support item assignment

In [123]:
'asdfadsf'[0]='b'

TypeError: 'str' object does not support item assignment

In [47]:
whereAreThey

[(5, 9), (1, 14), (16, 23), (0, 25)]

In [8]:
where_is_close

[9, 14, 23, 25]

In [ ]:
num_c = 

In [20]:
for i in pd_read['Compound'][0].split(pd_read_atoms["symboll"][85]):
    print(i.split(pd_read_atoms["symboll"][63]))

['[']
['5', '11N']
['', '2N', '3][']
['', '3N', '3]6Pb7I22']


In [22]:
for i in range(len_pd_read_keys):
    if i < len_pd_read_keys-1:
        x_np_array = np.append(x_np_array, pd_read[ pd_read_keys[i] ])
    if i == len_pd_read_keys-1:
        y_np_array = np.append(y_np_array, pd_read[ pd_read_keys[i] ])


In [23]:
x_np_array

array(['[C5H11NCH2NH3][CH3NH3]6Pb7I22', '[C(NH2)3][CH3NH3]3Pb3I10',
       '[CH3(CH2)3NH3]2[CH3NH3]6Pb7I22', '[CH3(CH2)2NH3]2[CH3NH3]2Pb3I10',
       '[CH3(CH2)3NH3]2[CH3NH3]5Pb6I19', '[CH3(CH2)3NH3]2[CH3NH3]4Pb5I16',
       '(CH3(CH2)3NH3)2(CH3NH3)Sn2I7', '[C5H4NCH2NH3][CH3NH3]3Pb4I13',
       '[C3H4N2(CH2)2NH3]SnI4',
       '[FC6H5(CH)2C6H5(CH2)2NH3]2_x000D_\n[CH3NH3]Sn2I7',
       '[NH3(CH2)8NH3][CH3NH3]3Pb4I14', '[C5H4NCH2NH3][CH3NH3]2Pb3I10',
       '[(CH3(CH2)4NH3)2(CH3NH3)4]Pb5I16', '[C(NH2)3][CH3NH3]2Pb2I7',
       '[C6H5CH2NH3]2[CH3NH3]2Pb3I10', '[C8H8N4]SnI4',
       '[CH3(CH2)2NH3]2[CH3NH3]3Pb4I13', '[NH3(CH)9NH3][CH3NH3]2Pb3I10',
       '[NH3(CH2)8NH3][CH3NH3]2Pb3I10',
       '[(CH3(CH2)4NH3)2(CH3NH3)3]Pb4I13',
       '[(CH3(CH2)5NH3)2(CH3NH3)3]Pb4I13', '[(CH3)3N(CH2)2NH3]SnI4',
       '[C6H5CH2NH3]2SnI4', '[CH3(CH2)11NH3]PbI3',
       '[C5H11NCH2NH3][CH3NH3]3Pb4I13', '[C6H5(CH2)2NH3]2SnI4',
       '[(2,3-C4H4S(CH3)-2,5-C4H4S)2(CH2)2NH3]2SnI4',
       '[C6F5(CH2)2NH3]_x000D

In [ ]:
x_np_array = x_np_array.reshape(6, 136).T

In [ ]:

x_np_array = x_np_array.reshape(6, 136).T



X_train, X_test, y_train, y_test = train_test_split(x_np_array, y_np_array, train_size=0.8, shuffle=True)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


In [ ]:
# back up code

# this is for counting

#  for k in range(len(pd_read)):
#     print(pd_read['Compound'][k])
#     compound_list = []
#     for i in pd_read['Compound'][k].split('['):
#         for j in i.split(']'):
#             for m in j.split('('):
#                 for n in m.split(')'):
#                     if j!='':
#                         #print(j)  
#                         compound_list.append(n)

#     #print(compound_list)
#     compound_list_with_n = []
#     for i in range(len(compound_list)-1):
#         # print(i)
#         compound_list_with_n.append(compound_list[i])
#         compound_list_with_n.append( find_first_num( compound_list[i+1] ) )
#     compound_list_with_n.append(compound_list[i+1])
#     compound_list_with_n.append( 1 )
#     print( compound_list_with_n )


#     count_element_num_a_sum = {}

#     for j in range(0,len(compound_list_with_n),2):
#         count_element_num_a = count_element_num( compound_list_with_n[j] )
#         if j==0:
#             for i in count_element_num_a:
#                 count_element_num_a_sum[i]=count_element_num_a[i]*compound_list_with_n[j+1]
#         else:
#             for i in count_element_num_a:
#                 count_element_num_a_sum[i] = count_element_num_a_sum[i] + count_element_num_a[i]*compound_list_with_n[j+1]

#         # for i in count_element_num_a:
#         #     if count_element_num_a[i]!=0:
#                 #print(i, count_element_num_a[i])

#     #print('')

#     for i in count_element_num_a_sum:
#         if count_element_num_a_sum[i]!=0:
#             print(i, count_element_num_a_sum[i])
